<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Фактуры:</h2>

----

### Вариант 10


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----
Создать базовый класс Invoice в C#, который будет представлять информацию о
фактурах за поставленные товары или оказанные услуги. На основе этого класса
разработать 2-3 производных класса, демонстрирующих принципы наследования и
полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и
методы, а также переопределены некоторые методы базового класса для
демонстрации полиморфизма.


#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
public class LineItem
{
    private string _name;
    private decimal _price;

    public string Name
    {
        get => _name;
        set => _name = string.IsNullOrWhiteSpace(value) ? "Без названия" : value;
    }

    public decimal Price
    {
        get => _price;
        set => _price = value < 0 ? 0 : value;
    }

    // Конструктор по умолчанию
    public LineItem()
    {
        Name = "Новая позиция";
        Price = 0;
    }

    // Конструктор с параметрами: присвоение строго через Свойства (сеттеры)
    public LineItem(string name, decimal price)
    {
        Name = name;   // Вызывает сеттер Name для валидации
        Price = price; // Вызывает сеттер Price для валидации
    }
}

public class Invoice
{
    private string _invoiceNumber;
    private DateTime _issueDate;
    private decimal _totalAmount;

    public string InvoiceNumber
    {
        get => _invoiceNumber;
        set => _invoiceNumber = string.IsNullOrWhiteSpace(value) ? "INV-000" : value;
    }

    public DateTime IssueDate
    {
        get => _issueDate;
        set => _issueDate = value;
    }

    public decimal TotalAmount
    {
        get => _totalAmount;
        protected set => _totalAmount = value < 0 ? 0 : value;
    }

    public List<LineItem> Items { get; protected set; } = new List<LineItem>();

    public Invoice()
    {
        InvoiceNumber = "INV-000";
        IssueDate = DateTime.Now;
    }

    // Конструктор с параметрами (использует сеттеры свойств)
    public Invoice(string number, DateTime date) : this()
    {
        InvoiceNumber = number; // Сеттер
        IssueDate = date;       // Сеттер
    }

    public virtual decimal CalculateTotal()
    {
        TotalAmount = Items.Sum(item => item.Price);
        return TotalAmount;
    }

    public virtual void AddLine(LineItem lineItem)
    {
        if (lineItem == null) return;
        Items.Add(lineItem);
        Console.WriteLine($"[Счёт {InvoiceNumber}] Добавлена позиция: {lineItem.Name} — {lineItem.Price} руб.");
    }

    public virtual void RemoveLine(LineItem lineItem)
    {
        if (lineItem != null && Items.Remove(lineItem))
        {
            Console.WriteLine($"[Счёт {InvoiceNumber}] Удалена позиция: {lineItem.Name}");
        }
    }

    // Взаимодействие 1: Перенос объекта LineItem в другой счёт
    public void SendItemTo(LineItem item, Invoice target)
    {
        if (item == null || target == null || target == this) return;

        if (Items.Contains(item))
        {
            RemoveLine(item);
            target.AddLine(item);
            Console.WriteLine($"--> [Взаимодействие] Позиция '{item.Name}' перенесена из '{InvoiceNumber}' в '{target.InvoiceNumber}'");
        }
    }

    // Взаимодействие 2: Слияние счетов
    public void MergeWith(Invoice other)
    {
        if (other == null || other == this) return;

        Console.WriteLine($"\n--> [Взаимодействие] Слияние счёта '{other.InvoiceNumber}' в счёт '{this.InvoiceNumber}'...");
        List<LineItem> toMove = new List<LineItem>(other.Items);
        foreach (LineItem line in toMove)
        {
            other.SendItemTo(line, this);
        }
    }

    // Взаимодействие 3: Сравнение двух счетов
    public void CompareTotalWith(Invoice other)
    {
        if (other == null) return;

        decimal myTotal = this.CalculateTotal();
        decimal otherTotal = other.CalculateTotal();

        Console.WriteLine($"\n--> [Взаимодействие] Сравнение счетов '{this.InvoiceNumber}' ({myTotal} руб.) и '{other.InvoiceNumber}' ({otherTotal} руб.):");
        if (myTotal > otherTotal)
            Console.WriteLine($"    Счёт '{this.InvoiceNumber}' больше на {myTotal - otherTotal} руб.");
        else if (myTotal < otherTotal)
            Console.WriteLine($"    Счёт '{other.InvoiceNumber}' больше на {otherTotal - myTotal} руб.");
        else
            Console.WriteLine("    Суммы счетов равны.");
    }
}

public class GoodsInvoice : Invoice
{
    private DateTime _supplyDate;

    public DateTime SupplyDate
    {
        get => _supplyDate;
        set => _supplyDate = value;
    }

    public GoodsInvoice(string number, DateTime date, DateTime supplyDate) : base(number, date)
    {
        SupplyDate = supplyDate; // Вызов сеттера
    }

    public override void AddLine(LineItem lineItem)
    {
        base.AddLine(lineItem);
        Console.WriteLine("    --> [Товар] Дата поставки: " + SupplyDate.ToShortDateString());
    }
}

public class ServiceInvoice : Invoice
{
    private DateTime _serviceDate;

    public DateTime ServiceDate
    {
        get => _serviceDate;
        set => _serviceDate = value;
    }

    public ServiceInvoice(string number, DateTime date, DateTime serviceDate) : base(number, date)
    {
        ServiceDate = serviceDate; // Вызов сеттера
    }

    public override void RemoveLine(LineItem lineItem)
    {
        base.RemoveLine(lineItem);
        Console.WriteLine("    --> [Услуга] Причина удаления: Отмена услуги клиентом.");
    }
}

public class CombinedInvoice : Invoice
{
    private bool _returnAllowed;

    public bool ReturnAllowed
    {
        get => _returnAllowed;
        set => _returnAllowed = value;
    }

    public CombinedInvoice(string number, DateTime date, bool returnAllowed) : base(number, date)
    {
        ReturnAllowed = returnAllowed; // Вызов сеттера
    }

    public override decimal CalculateTotal()
    {
        decimal total = base.CalculateTotal();
        if (ReturnAllowed)
        {
            total += total * 0.05m;
        }
        TotalAmount = total;
        return TotalAmount;
    }
}

// =====================================================
// 2. ИСПОЛНЯЕМЫЙ КОД (Выполняется прямо в Notebook)
// =====================================================

Console.WriteLine("=== 1. СОЗДАНИЕ ОБЪЕКТОВ И РАБОТА СЕТТЕРОВ В КОНСТРУКТОРЕ ===");

// Позиции
LineItem laptop = new LineItem("Ноутбук", 50000);
LineItem mouse = new LineItem("Мышь", 1200);
LineItem consulting = new LineItem("Консультация", 8000);
LineItem setup = new LineItem("Настройка ПО", 2500);

// Счета
GoodsInvoice goodsInv = new GoodsInvoice("Т-001", DateTime.Now, DateTime.Now.AddDays(3));
ServiceInvoice serviceInv = new ServiceInvoice("У-001", DateTime.Now, DateTime.Now.AddDays(1));
CombinedInvoice combinedInv = new CombinedInvoice("К-001", DateTime.Now, true);

// Наполнение
goodsInv.AddLine(laptop);
goodsInv.AddLine(mouse);
serviceInv.AddLine(consulting);
combinedInv.AddLine(setup);

Console.WriteLine("\n=== 2. ДЕМОНСТРАЦИЯ ВЗАИМОДЕЙСТВИЯ ОБЪЕКТОВ ===");

// 1) Перенос позиций между объектами
goodsInv.SendItemTo(mouse, serviceInv);

// 2) Сравнение двух объектов-счетов
goodsInv.CompareTotalWith(serviceInv);

// 3) Слияние одного счета в другой
combinedInv.MergeWith(serviceInv);

Console.WriteLine("\n=== 3. ИТОГОВЫЕ РАСЧЁТЫ ===");
Console.WriteLine($"Итого 'Товарный счёт' ({goodsInv.InvoiceNumber}): {goodsInv.CalculateTotal()} руб.");
Console.WriteLine($"Итого 'Счёт услуг' ({serviceInv.InvoiceNumber}): {serviceInv.CalculateTotal()} руб.");
Console.WriteLine($"Итого 'Комбинированный счёт' ({combinedInv.InvoiceNumber}) [с 5% наценкой]: {combinedInv.CalculateTotal()} руб.");

=== 1. СОЗДАНИЕ ОБЪЕКТОВ И РАБОТА СЕТТЕРОВ В КОНСТРУКТОРЕ ===
[Счёт Т-001] Добавлена позиция: Ноутбук — 50000 руб.
    --> [Товар] Дата поставки: 08.10.2026
[Счёт Т-001] Добавлена позиция: Мышь — 1200 руб.
    --> [Товар] Дата поставки: 08.10.2026
[Счёт У-001] Добавлена позиция: Консультация — 8000 руб.
[Счёт К-001] Добавлена позиция: Настройка ПО — 2500 руб.

=== 2. ДЕМОНСТРАЦИЯ ВЗАИМОДЕЙСТВИЯ ОБЪЕКТОВ ===
[Счёт Т-001] Удалена позиция: Мышь
[Счёт У-001] Добавлена позиция: Мышь — 1200 руб.
--> [Взаимодействие] Позиция 'Мышь' перенесена из 'Т-001' в 'У-001'

--> [Взаимодействие] Сравнение счетов 'Т-001' (50000 руб.) и 'У-001' (9200 руб.):
    Счёт 'Т-001' больше на 40800 руб.

--> [Взаимодействие] Слияние счёта 'У-001' в счёт 'К-001'...
[Счёт У-001] Удалена позиция: Консультация
    --> [Услуга] Причина удаления: Отмена услуги клиентом.
[Счёт К-001] Добавлена позиция: Консультация — 8000 руб.
--> [Взаимодействие] Позиция 'Консультация' перенесена из 'У-001' в 'К-001'
[Счёт У-001] Удален